# Lab 5 — Exam extras: Delta maintenance, SQL patterns, streaming (beyond course material)
Lessons 5.2, 5.4, 5.5 · exercises `5a`–`5d` · schema `lab5`

**How this lab works**
- Run cells top to bottom (`Shift+Enter`). After a kernel restart, run again from the top.
- Cells marked `# TODO` contain `<FILL-IN>`: replace it with your code, then run the `check(...)` cell.
- Stuck? Ask Claude in the terminal: *"hint for 1c"*. Solutions are in `notebooks/solutions/` (try first!).
- 🖥 **Local** means open-source Spark + Delta on your Mac. ☁ **Databricks** shows the real syntax for the exam.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # lets the notebook import labutils
from labutils import get_spark, path, reset_lab, show, checkpoint, table_path, landing
from labutils.check import check
from pyspark.sql import functions as F
spark = get_spark()
reset_lab("lab5")

## 5a · OPTIMIZE, RESTORE, VACUUM (lesson 5.2)
Many small writes → many small files → slow reads. `OPTIMIZE` compacts them (`ZORDER BY` co-locates by column).
`RESTORE` rolls back to an old version. `VACUUM` deletes files not used by the current version (default keep 7 days) —
after it, time travel to old versions **fails** (you will see red Spark error logs — that is expected).

In [ ]:
spark.sql("CREATE TABLE lab5.events_small (id INT, kind STRING)")
for i in range(12):
    spark.sql(f"INSERT INTO lab5.events_small VALUES ({i}, '{'click' if i % 2 else 'view'}')")
def files():
    return spark.sql("DESCRIBE DETAIL lab5.events_small").first().numFiles
print("files before OPTIMIZE:", files())

In [ ]:
# TODO 5a
spark.sql("<FILL-IN 1>")
print("files after OPTIMIZE:", files())
spark.sql("<FILL-IN 2>")
print("rows after RESTORE:", spark.table("lab5.events_small").count())
spark.sql("<FILL-IN 3>")

In [ ]:
show(spark.sql("DESCRIBE HISTORY lab5.events_small").select("version", "operation"))
try:
    # note: count(*) alone can be answered from the Delta log stats, so read real data:
    spark.sql("SELECT * FROM lab5.events_small VERSION AS OF 5").collect()
except Exception as e:
    print("Time travel after VACUUM fails:", type(e).__name__)
check("5a")

## 5b · Latest row per key with a window function (lesson 5.4)
Very common exam pattern: `row_number() OVER (PARTITION BY key ORDER BY ts DESC)` then keep `rn = 1`.

In [ ]:
spark.sql("""CREATE OR REPLACE TEMP VIEW user_updates AS SELECT * FROM VALUES
  ('u1', 'u1@first.com',  TIMESTAMP'2024-01-01 10:00'),
  ('u1', 'u1@third.com',  TIMESTAMP'2024-03-01 10:00'),
  ('u1', 'u1@second.com', TIMESTAMP'2024-02-01 10:00'),
  ('u2', 'u2@x.com',      TIMESTAMP'2024-01-05 10:00'),
  ('u3', 'u3@x.com',      TIMESTAMP'2024-01-07 10:00'),
  ('u3', 'u3@x.com',      TIMESTAMP'2024-01-07 10:00')
  AS t(user_id, email, updated_at)""")

In [ ]:
# TODO 5b
spark.sql("""
CREATE OR REPLACE TABLE lab5.users_latest AS
WITH ranked AS (
  SELECT *, <FILL-IN 1> AS rn FROM user_updates)
SELECT user_id, email, updated_at FROM ranked WHERE rn = 1
""")

In [ ]:
show(spark.table("lab5.users_latest")); check("5b")

## 5c · Higher-order functions on arrays (lesson 5.4)
Work on arrays **without exploding**: `size`, `transform(arr, x -> ...)`, `filter(arr, x -> ...)`,
`exists(arr, x -> ...)`, `aggregate(arr, start, (acc, x) -> ...)`.

In [ ]:
schema = "STRUCT<user_id: STRING, items: ARRAY<STRUCT<item_id: STRING, price_usd: DOUBLE, quantity: BIGINT>>>"
spark.sql(f"""CREATE OR REPLACE TEMP VIEW events AS
SELECT from_json(cast(unbase64(value) AS STRING), '{schema}') AS e FROM json.`{path('events-kafka')}`""")

In [ ]:
# TODO 5c
spark.sql("""
CREATE OR REPLACE TABLE lab5.event_item_totals AS
SELECT e.user_id,
       <FILL-IN 1> AS item_count,          -- number of items
       <FILL-IN 2> AS order_value,         -- sum of price_usd * quantity (aggregate)
       <FILL-IN 3> AS expensive_items      -- items with price_usd > 1000 (filter)
FROM events
""")

In [ ]:
show(spark.table("lab5.event_item_totals"), 8); check("5c")

## 5d · Streaming aggregation and output modes (lesson 5.5)
Output modes: **append** (only new rows — not allowed for aggregations without a watermark), **complete** (rewrite the
whole result), **update** (only changed rows). **Task:** pick the output mode that writes the full count table.

In [ ]:
landing.reset("orders_agg"); landing.land("orders_agg", 3)
orders_schema = "order_id BIGINT, order_timestamp STRING, customer_id BIGINT, notifications STRING"

In [ ]:
# TODO 5d
(spark.readStream.format("json").schema(orders_schema).load(landing.volume("orders_agg"))
      .groupBy("customer_id").agg(F.count("*").alias("n_orders"))
 .writeStream.outputMode("<FILL-IN 1>")
      .option("checkpointLocation", checkpoint("lab5", "orders_per_customer"))
      .trigger(availableNow=True).toTable("lab5.orders_per_customer").awaitTermination())

In [ ]:
show(spark.table("lab5.orders_per_customer").orderBy(F.desc("n_orders")), 5); check("5d")